Importing Necessary Libraries

In [ ]:
import pandas as pd
from datetime import datetime

Loading Dataset

In [17]:
df = pd.read_json("../data/sampled_dataset.json")
df.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,AO94DHGC771SJ,0528881469,amazdnu,"[0, 0]",We got this GPS for my husband who is an (OTR)...,5,Gotta have GPS!,1370131200,"06 2, 2013"
1,AMO214LNFCEI4,0528881469,Amazon Customer,"[12, 15]","I'm a professional OTR truck driver, and I bou...",1,Very Disappointed,1290643200,"11 25, 2010"
2,A3N7T0DY83Y4IG,0528881469,C. A. Freeman,"[43, 45]","Well, what can I say. I've had this unit in m...",3,1st impression,1283990400,"09 9, 2010"
3,A1H8PY3QHMQQA0,0528881469,"Dave M. Shaw ""mack dave""","[9, 10]","Not going to write a long review, even thought...",2,"Great grafics, POOR GPS",1290556800,"11 24, 2010"
4,A24EV6RXELQZ63,0528881469,Wayne Smith,"[0, 0]",I've had mine for a year and here's what we go...,1,"Major issues, only excuses for support",1317254400,"09 29, 2011"


Dataset Inspection

In [18]:
df.shape
df.columns
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   reviewerID      40000 non-null  object
 1   asin            40000 non-null  object
 2   reviewerName    39733 non-null  object
 3   helpful         40000 non-null  object
 4   reviewText      40000 non-null  object
 5   overall         40000 non-null  int64 
 6   summary         40000 non-null  object
 7   unixReviewTime  40000 non-null  int64 
 8   reviewTime      40000 non-null  object
dtypes: int64(2), object(7)
memory usage: 2.7+ MB


reviewerID          0
asin                0
reviewerName      267
helpful             0
reviewText          0
overall             0
summary             0
unixReviewTime      0
reviewTime          0
dtype: int64

In [19]:
df.drop(columns=['reviewerName'], inplace=True)

The reviewerName column was removed due to high missing value ratio (267 nulls) and redundancy, as reviewerID provides a unique identifier sufficient for user-level analysis and recommendation modeling.

In [20]:
df.columns

Index(['reviewerID', 'asin', 'helpful', 'reviewText', 'overall', 'summary',
       'unixReviewTime', 'reviewTime'],
      dtype='object')

In [21]:
df['helpful'] = df['helpful'].apply(tuple)
df = df.drop_duplicates()

Converting DataTypes

In [22]:
df['overall'] = df['overall'].astype(float)

In [23]:
df['reviewTime'] = pd.to_datetime(df['reviewTime'], errors='coerce')

Feature Engineering

In [24]:
def sentiment(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"

df['sentiment'] = df['overall'].apply(sentiment)

In [25]:
def action(score):
    if score == 5:
        return "purchase"
    elif score == 4:
        return "add_to_cart"
    elif score == 3:
        return "view"
    else:
        return "ignore"

df['user_action'] = df['overall'].apply(action)

In [26]:
df['review_length'] = df['reviewText'].apply(lambda x: len(str(x).split()))

In [27]:
df['helpful_ratio'] = df['helpful'].apply(
    lambda x: x[0] / x[1] if x[1] != 0 else 0
)

In [28]:
df['year'] = df['reviewTime'].dt.year

In [29]:
cleaned_data = df[[
    "reviewerID",
    "asin",
    "reviewText",
    "summary",
    "overall",
    "sentiment",
    "user_action",
    "review_length",
    "helpful_ratio",
    "year"
]]

In [30]:
# 1. Save with lines=True (Fast & Big Data Friendly)
df.to_json("../data/cleaned_dataset.json", orient="records", lines=True)

# 2. Read with lines=True (This avoids the EOF / Trailing data error)
df = pd.read_json("../data/cleaned_dataset.json", lines=True)


In [31]:
print(df.shape)

(40000, 13)
